In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!nvidia-smi || true

!pip install -U "transformers==4.57.3"

Mounted at /content/drive
Thu Jan  1 15:11:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

# The Dirve Path

In [ ]:
from pathlib import Path

HF_HOME = Path("/content/drive/MyDrive/hf_home")          # HF 全局缓存（权重/processor 等）
MODEL_DIR = Path("/content/drive/MyDrive/hf_models/Qwen2.5-Omni-7B")  # 单独放这个模型（可选）
ROOT_DIR  = Path("./data/")    # 你的 xlsx 所在目录
BASE_AUDIO_DIR = "./data/materials_updated/audio/withblank/"     # 音频根目录
BASE_IMAGE_DIR = "./data/materials_updated/visual/"     # 图片根目录

HF_HOME.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DIR.mkdir(parents=True, exist_ok=True)

import os
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")

# Predownload

optional

In [ ]:
#from huggingface_hub import snapshot_download

#repo_id = "Qwen/Qwen2.5-Omni-7B"  # 或你要用的具体 omni 变体
#snapshot_download(
    #repo_id=repo_id,
    #local_dir=str(MODEL_DIR),
    #local_dir_use_symlinks=False,
    #resume_download=True,
    #revision=None  # 需要固定版本可写 tag/commit
#)
#print("Model snapshot is ready at:", MODEL_DIR)

In [ ]:
# ==== inference_colab.py (单元内定义, 直接运行生效) ====
import os
from pathlib import Path
from typing import Optional, List

import pandas as pd
import torch
from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)

# ================== 简单配置（改这里） ==================
ROOT_DIR = Path("./data/")     # 你的 .xlsx 所在目录
SHEET_NAME: Optional[str] = None     # None=默认第一个工作表
MODEL_ID_OR_PATH = "/content/drive/MyDrive/hf_models/Qwen2.5-Omni-7B"      # 本地已下载的模型目录

# 路径前缀（字符串；可留空""）
# BASE_AUDIO_DIR = "./data/materials/audio/withblank/"
# BASE_IMAGE_DIR = "./data/materials/visual/"

MAX_NEW_TOKENS = 64
TEMPERATURE = 1
TOP_P = 0.9
N_RUNS = 20
SEEDS: List[int] = [11,23,37,41,53,67,79,83,97,101,113,127,131,149,163,173,181,197,211,223]
COL_PREFIX = "Omni_"

PROMPT_TEMPLATE = (
"""你将看到听到一个句子的前半部分，同时看到一张 2×2 画布（尺寸 1024×768，原点左上，x 向右，y 向下）。
画布被划分为四个区域，它们的像素边界为：
  TL: [106, 42, 406, 342]
	BL: [106, 426, 406, 726]
	BR: [618, 426, 918, 726]
	TR: [618, 42, 918, 342]
你的任务：
	1.	根据听到的句子前半部分，从四个象限中选择最有可能在接下来被提及的一个图（注意：不是已经被提及的）。
	2.	根据你选择的象限中可见内容，补全一句最小化的续写词/短语，使整句自然通顺；不得重复已听到的内容，不得引入画面之外的新名词。续写应与所选象限内容一致（如：“水果”）。
约束：
	quadrant 必须是四选一；bbox_px 必须与该象限的边界一致；
	continuation 必须来自所选象限可见的人/物/动作/状态/场景；
	不得出现集合外的新实体；不得复述前导内容。
以下是音频和图片：：<image><audio>
输出格式（必须严格遵守，仅输出 JSON，不要额外文字）：
{
  "quadrant": "TL|TR|BL|BR",
  "bbox_px": [x1, y1, x2, y2],
  "continuation": "<你的续写词/短语>"
}"""
)
# =======================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [ ]:
def load_model_and_processor():
    # bfloat16 对 A100/H100 友好；否则退回 float16
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

    model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=dtype,
        device_map="auto" if DEVICE.type == "cuda" else None,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )
    if DEVICE.type != "cuda":
        model = model.to(DEVICE)

    processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID_OR_PATH)
    return model, processor

model, processor = load_model_and_processor()

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [ ]:


def build_content_from_prompt(template: str, image_path: Path, audio_path: Path):
    before, after_img = template.split("<image>")
    mid, after = after_img.split("<audio>")

    content = []
    if before.strip():
        content.append({"type": "text", "text": before})
    content.append({"type": "image", "path": str(image_path)})
    if mid.strip():
        content.append({"type": "text", "text": mid})
    content.append({"type": "audio", "path": str(audio_path)})
    if after.strip():
        content.append({"type": "text", "text": after})
    return content

def build_inputs(processor, model, content):
    conversations = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        padding=True,
    )
    # 放到设备
    if DEVICE.type == "cuda":
        return {k: v.to("cuda") for k, v in inputs.items()}
    else:
        return {k: v.to(DEVICE) for k, v in inputs.items()}

@torch.inference_mode()
def generate_once(model, processor, inputs, seed: int):
    torch.manual_seed(int(seed))
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(int(seed))
    out_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    return processor.batch_decode(out_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

def make_path(xlsx_dir: Path, base_dir_str: str, p_str: str) -> Path:
    # 极简规则：绝对路径则用绝对路径；否则优先 BASE_*_DIR，不填就用 xlsx_dir
    p = Path(p_str)
    if p.is_absolute():
        return p
    if base_dir_str:
        return (Path(base_dir_str) / p).resolve()
    return (xlsx_dir / p).resolve()

def process_one_excel(xlsx_path: Path, model, processor, output_path="./data/gpt2/updated_auWS/TR_last"):
    print(f"\n>>> 处理：{xlsx_path.name}")
    df = pd.read_excel(xlsx_path, engine="openpyxl") if SHEET_NAME is None \
        else pd.read_excel(xlsx_path, sheet_name=SHEET_NAME, engine="openpyxl")

    # 只认这三个列名
    for c in ["Item", "Audio_File_new", "Image_File"]:
        if c not in df.columns:
            raise ValueError(f"{xlsx_path.name} 缺少列：{c}")

    out_cols = [f"{COL_PREFIX}{i:02d}" for i in range(1, N_RUNS+1)]
    for c in out_cols:
        if c not in df.columns:
            df[c] = ""

    xlsx_dir = xlsx_path.parent

    for idx, row in df.iterrows():
        item = row["Item"]
        audio_path = make_path(xlsx_dir, BASE_AUDIO_DIR, str(row["Audio_File_new"]).strip())
        image_path = make_path(xlsx_dir, BASE_IMAGE_DIR, str(row["Image_File"]).strip())

        if not audio_path.exists():
            print(f"  [跳过] Item={item} 音频不存在: {audio_path}")
            continue
        if not image_path.exists():
            print(f"  [跳过] Item={item} 图片不存在: {image_path}")
            continue

        content = build_content_from_prompt(PROMPT_TEMPLATE, image_path, audio_path)
        inputs = build_inputs(processor, model, content)

        for run_i in range(N_RUNS):
            col = out_cols[run_i]
            if str(df.at[idx, col]).strip():  # 断点续跑：已有就跳过
                continue
            seed = SEEDS[run_i % len(SEEDS)]
            try:
                out_text = generate_once(model, processor, inputs, seed)
                # —— 这里打印模型真实输出（实时反馈）——
                print(f"[{xlsx_path.name}] Item={item} Run {run_i+1}/{N_RUNS} seed={seed} -> {out_text}", flush=True)
                df.at[idx, col] = out_text
            except Exception as e:
                err = f"[GEN_ERROR seed={seed}] {e}"
                print(f"[{xlsx_path.name}] Item={item} Run {run_i+1}/{N_RUNS} seed={seed} -> {err}", flush=True)
                df.at[idx, col] = err
        print(f"  完成 Item={item}")

    output_dir = Path(output_path)
# 确保目录存在
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{xlsx_path.stem}__omni20.xlsx"
    df.to_excel(out_path, index=False)
    print(f"<<< 写入：{out_path.name}")


xlsx_files = sorted([p for p in ROOT_DIR.glob("*.xlsx")
                      if not p.name.startswith("~$") and "__omni20" not in p.stem])
if not xlsx_files:
    print(f"在 {ROOT_DIR.resolve()} 下未找到待处理的 .xlsx")
    exit()
print(f"[ROOT_DIR] {ROOT_DIR}")
print(f"[BASES] audio='{BASE_AUDIO_DIR}' image='{BASE_IMAGE_DIR}'")
for x in xlsx_files:
    process_one_excel(x, model, processor)
